In [113]:
import pandas as pd
import numpy as np
import plotly.express as px

RUTA = "../data/processed/"
RADIO = 300
TAMANIO_CUADRO = 0.0027

victimas_camaras_anio = "AA_Acc"
victimas_camaras_dist = "distancia_m"
victimas_camaras_tipo = "Condicion"
victimas_camaras_franja = "franja"
victimas_camaras_lat = "Latitud"
victimas_camaras_lon = "Longitud"
victimas_camaras_cuadro = "cuadro"

cuadros_nombre = "cuadro"
cuadros_nvict = "victimas"
cuadros_dist = "distancia_min"

localidades_nombre = "localidad"
localidades_tasa = "tasa_victimas"
localidades_cam = "camaras"

In [114]:
victimas_camaras = pd.read_csv(RUTA + "victimas_camaras.csv.gz")
cuadros = pd.read_csv(RUTA + "cuadros.csv")
localidades = pd.read_csv(RUTA + "localidades.csv")

fila = np.round(victimas_camaras[victimas_camaras_lat] / TAMANIO_CUADRO).astype(int)
columna = np.round(victimas_camaras[victimas_camaras_lon] / TAMANIO_CUADRO).astype(int)
victimas_camaras[victimas_camaras_cuadro] = fila.astype(str) + "_" + columna.astype(str)

victimas_camaras_2019 = victimas_camaras[victimas_camaras[victimas_camaras_anio] >= 2019]

print(victimas_camaras_2019[victimas_camaras_cuadro].value_counts().head(3))
print(cuadros[[cuadros_nombre, cuadros_nvict]].head(3))

cuadro
1714_-27448    260
1703_-27474    257
1703_-27461    255
Name: count, dtype: int64
        cuadro  victimas
0  1714_-27448       260
1  1703_-27474       257
2  1703_-27461       255


In [115]:
print(cuadros.shape)
print(cuadros.dtypes)
print(cuadros.head(3))

(3925, 12)
cuadro                  str
victimas              int64
muertos               int64
latitud             float64
longitud            float64
distancia_min       float64
localidad               str
direccion               str
actor_principal         str
franja_principal        str
tiene_camara           bool
puesto                int64
dtype: object
        cuadro  victimas  muertos   latitud   longitud  distancia_min  \
0  1714_-27448       260        7  4.627833 -74.109302     412.076357   
1  1703_-27474       257        5  4.597461 -74.179590     294.675102   
2  1703_-27461       255       10  4.598402 -74.144987     420.920019   

       localidad                               direccion actor_principal  \
0  PUENTE ARANDA  AV AVENIDA DE LAS AMERICAS - KR 53  02    MOTOCICLISTA   
1           BOSA                      AU SUR - KR 76  02    MOTOCICLISTA   
2        KENNEDY          AV AVENIDA BOYACA - CL 44 S 02    MOTOCICLISTA   

  franja_principal  tiene_camara  puesto 

In [116]:
print(localidades.shape)
print(localidades.dtypes)
print(localidades.head(3))

(20, 8)
localidad                  str
victimas                 int64
muertos                  int64
poblacion_suma           int64
tasa_victimas          float64
tasa_muertos           float64
camaras                float64
victimas_por_camara    float64
dtype: object
        localidad  victimas  muertos  poblacion_suma  tasa_victimas  \
0    LOS MÁRTIRES      4756      120          581791     817.475691   
1  ANTONIO NARIÑO      3708       88          581749     637.388289   
2        SANTA FE      4750      140          752232     631.454126   

   tasa_muertos  camaras  victimas_por_camara  
0     20.625964      0.0                  NaN  
1     15.126799      1.0               3708.0  
2     18.611279      2.0               2375.0  


In [117]:
pct_cerca = (victimas_camaras[victimas_camaras_dist] <= RADIO).mean() * 100
pct_lejos = 100 - pct_cerca
print(f"A {RADIO} m o menos: {pct_cerca:.1f}%  |  Más lejos: {pct_lejos:.1f}%")

g1 = pd.DataFrame([[f"≤ {RADIO} m", pct_cerca], [f"> {RADIO} m", pct_lejos]],
                  columns=["zona", "pct"])

fig1 = px.bar(g1, x="zona", y="pct", text_auto=".1f", color="zona",
              title="Víctimas según distancia a la cámara más cercana")
fig1.update_layout(yaxis_title="% de las víctimas", showlegend=False, title_x=0.5, xaxis_title="Distancia a la cámara más cercana")
fig1.show()

A 300 m o menos: 8.8%  |  Más lejos: 91.2%


In [118]:
g2 = cuadros.head(10)
g2["Cámara"] = np.where(g2[cuadros_dist] <= RADIO, "Con cámara", "Sin cámara")
g2[cuadros_nombre] = g2[cuadros_nombre].astype(str)

fig2 = px.bar(g2, x=cuadros_nvict, y=cuadros_nombre, color="Cámara", orientation="h",
              title="  10 puntos con más víctimas")
fig2.update_yaxes(autorange="reversed")
fig2.update_layout(title_x=0.5)
fig2.show()

In [119]:
top50 = cuadros.head(50)
puntos_sin = top50[top50[cuadros_dist] > RADIO]
vic_sin = victimas_camaras_2019[victimas_camaras_2019[victimas_camaras_cuadro].isin(puntos_sin[cuadros_nombre])]
print("Puntos críticos sin cámara:", len(puntos_sin), "de 50")
print("Víctimas en esos puntos:", len(vic_sin))

g3a = vic_sin[victimas_camaras_tipo].value_counts(normalize=True).mul(100).reset_index()
g3a.columns = ["tipo", "pct"]

fig3a = px.bar(g3a, x="pct", y="tipo", orientation="h",
               title="Tipo de víctima en puntos críticos sin cámara")
fig3a.update_layout(xaxis_title="% de las víctimas", title_x=0.5, yaxis_title="Tipo de víctima")
fig3a.update_yaxes(autorange="reversed")
fig3a.show()

orden = ["Madrugada (0-5)", "Mañana (6-11)", "Tarde (12-17)", "Noche (18-23)"]
g3b = vic_sin[victimas_camaras_franja].value_counts(normalize=True).reindex(orden).mul(100).reset_index()
g3b.columns = ["franja", "pct"]

fig3b = px.bar(g3b, x="franja", y="pct",
               title="Franja horaria en puntos críticos sin cámara")
fig3b.update_layout(yaxis_title="% de las víctimas", title_x=0.5, xaxis_title="Franja horaria")
fig3b.show()

Puntos críticos sin cámara: 34 de 50
Víctimas en esos puntos: 7114


In [120]:
g4 = localidades.sort_values(localidades_tasa, ascending=False)

fig4a = px.bar(g4, x=localidades_tasa, y=localidades_nombre, orientation="h",
               title="Víctimas por 100.000 habitantes")
fig4a.update_yaxes(autorange="reversed")
fig4a.update_layout(yaxis_title="Localidad", xaxis_title="Víctimas por 100.000 habitantes", title_x=0.5)
fig4a.show()

fig4b = px.bar(g4, x=localidades_cam, y=localidades_nombre, orientation="h",
               title="Número de cámaras por localidad")
fig4b.update_yaxes(autorange="reversed")
fig4b.show()

In [121]:
g5 = victimas_camaras_2019.copy()
g5[victimas_camaras_anio] = g5[victimas_camaras_anio].astype(int)
g5["zona"] = np.where(g5[victimas_camaras_dist] <= RADIO, "Cerca de una cámara", "Resto de la ciudad")

por_anio = g5.groupby([victimas_camaras_anio, "zona"]).size().unstack(fill_value=0)
print(por_anio)

indice = (por_anio / por_anio.loc[2019] * 100).reset_index()
indice = indice.melt(id_vars=victimas_camaras_anio, var_name="zona", value_name="indice")

fig5 = px.line(indice, x=victimas_camaras_anio, y="indice", color="zona", markers=True,
               title="Víctimas por año (base 2019 = 100)")
fig5.add_hline(y=100, line_dash="dash")
fig5.update_layout(title_x=0.5)
fig5.show()

zona    Cerca de una cámara  Resto de la ciudad
AA_Acc                                         
2019                   1672               16180
2020                   1083               12006
2021                   1625               17213
2022                   2101               19445
2023                   2036               21496
2024                   2043               21111
2025                   1606               18561


In [122]:
g6 = victimas_camaras.copy()
g6[victimas_camaras_anio] = g6[victimas_camaras_anio].astype(int)

por_anio_total = g6.groupby(victimas_camaras_anio).size().reset_index()
por_anio_total.columns = ["anio", "victimas"]
por_anio_total["Período"] = np.where(por_anio_total["anio"] < 2019, "Antes de 2019", "Desde 2019")

promedio_antes = por_anio_total[por_anio_total["anio"] < 2019]["victimas"].mean()
promedio_despues = por_anio_total[por_anio_total["anio"] >= 2019]["victimas"].mean()
print(f"Promedio anual antes de 2019: {promedio_antes:.0f}  |  Desde 2019: {promedio_despues:.0f}")

fig6 = px.bar(por_anio_total, x="anio", y="victimas", color="Período", text_auto=True,
              title="Víctimas por año en toda la ciudad")
fig6.update_layout(xaxis_title="Año", yaxis_title="Víctimas", title_x=0.5)
fig6.update_xaxes(dtick=1)
fig6.show()

Promedio anual antes de 2019: 14963  |  Desde 2019: 19740


In [123]:
radios = [300, 500]
filas = []

for r in radios:
    pct = (victimas_camaras[victimas_camaras_dist] <= r).mean() * 100
    sin_r = top50[top50[cuadros_dist] > r]
    vic_r = victimas_camaras_2019[victimas_camaras_2019[victimas_camaras_cuadro].isin(sin_r[cuadros_nombre])]
    motos = (vic_r[victimas_camaras_tipo] == "MOTOCICLISTA").mean() * 100
    filas.append([r, pct, len(sin_r), motos])

comparacion = pd.DataFrame(filas, columns=["radio", "pct_cerca", "puntos_sin_camara", "pct_motos"])
print(comparacion.round(1))

   radio  pct_cerca  puntos_sin_camara  pct_motos
0    300        8.8                 34       50.2
1    500       17.8                 25       52.0
